# Laya vs Jev, judged on the same task
Same 500 items, same two questions, zero-shot. Every number below is computed live in this run.

In [1]:
from nb_helpers import *
versions()

laya           0.3.5
laya-mlx       0.2.0
typesafe-sdk   0.7.1
machine        arm64, 26.6.2


## The 500 items and the reference judge
Claude Sonnet graded every item blind, earlier. **Agreement** below means agreement with it, not correctness.

In [2]:
items = load_items()
corpus_summary(items)

items: 500
reference judge says rework: 63.4%   says none: 36.6%
reference judge vs what I planted: verdict 92.0%, exact defect 67.6%
reference judge cost: $5.36 per 1,000 items (measured tokens)


## Two questions, identical for both candidates

In [3]:
print(VERDICT_OPTIONS)
print(len(CRITERIA), 'failure modes, e.g.', list(CRITERIA)[1:6])

{'pass': 'the answer is factually correct, responsive to the question, and free of invented detail', 'rework': 'the answer is wrong, evasive, incomplete, or states something it cannot support'}
40 failure modes, e.g. ['factual_error', 'fabricated_citation', 'fabricated_number', 'outdated_fact', 'wrong_entity']


## Laya, local on Apple silicon
`laya-mlx` is a community runtime, not written by Convai.

In [4]:
laya = load_laya()
time_laya(laya, items)

typed-decisions checkpoint loaded in 0.5 s
one 2-option call, median of 50: 39.0 ms (local, no network)


In [5]:
laya_verdict = run_laya(laya, items, {'verdict': QUESTIONS['verdict']})
score(laya_verdict, items, 'verdict')

verdict: agreement with reference 0.382   p50 39.0 ms   top picks [('pass', 492), ('rework', 8)]


## Was I holding it wrong? Four framings, first 200 items

In [6]:
four_framings(laya, items[:200])

original, pass listed first    pass 197 / rework   3
flipped, rework listed first   pass 198 / rework   2
short labels                   pass 194 / rework   6
negated framing                pass 194 / rework   6


## The 40-option question, and my mistake

In [7]:
budget_check(laya)

shared budget for question + options (head_max_len): 256 tokens
my 40 options with descriptions want:                492 tokens
each option slot after trimming:                     6 to 6 tokens (1 is the marker)
instruction tokens kept:                             16 of 17


In [8]:
laya_criterion = criterion_three_ways(laya, items)

long descriptions, budget 256   criterion: agreement with reference 0.120   p50 70.5 ms   top picks [('none', 209), ('unsafe_advice', 78)]
short labels,      budget 256   criterion: agreement with reference 0.228   p50 71.9 ms   top picks [('none', 349), ('unsafe_advice', 77)]
long descriptions, budget 1024   criterion: agreement with reference 0.234   p50 117.3 ms   top picks [('none', 336), ('outside_expertise', 91)]


## Jev, hosted API

In [9]:
await jev_single_call(items)

one call, median of 10, sequential: 159 ms (hosted API, includes network)


In [10]:
jev = await run_jev(items)
score(jev, items, 'verdict'); score(jev, items, 'criterion')

verdict: agreement with reference 0.876   p50 161.6 ms   top picks [('rework', 257), ('pass', 243)]
criterion: agreement with reference 0.790   p50 161.6 ms   top picks [('none', 204), ('factual_error', 51)]


In [ ]:
jev_short = await run_jev_short(items)
score(jev_short, items, 'criterion')

## What Laya's confidence field actually is (laya 0.3.5)

In [ ]:
show_confidence_source()

## How much of the grading bill can I delete?
Accept the most confident answers unreviewed, send the rest to the reference judge.

In [ ]:
cascade_table({
    'Jev': (jev, jev['usd'] / len(items)),
    'Laya, 2 options': (laya_verdict, 0.0),
    'Laya, 40 options, budget 1024': (laya_criterion['long descriptions, budget 1024'], 0.0),
}, items)